# Day 16 - Feature Importance + Feature Selection


## 학습 목표
- Feature Importance 시각화
- SelectFromModel / PCA 기반 특성 선택


## 1. Feature Importance 분석


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score
import xgboost as xgb

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

# RandomForest Importance
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_tr, y_tr)
imp_rf = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 10))
imp_rf.tail(15).plot(kind='barh', color='steelblue')
plt.title('RandomForest Feature Importance (Top 15)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 2. SelectFromModel로 특성 선택


In [ ]:
selector = SelectFromModel(rf, threshold='median')
X_tr_sel = selector.fit_transform(X_tr, y_tr)
X_te_sel = selector.transform(X_te)
selected = feature_names[selector.get_support()]
print(f"Selected features ({len(selected)}/{len(feature_names)}):")
print(list(selected))

rf_sel = RandomForestClassifier(n_estimators=200, random_state=42)
rf_sel.fit(X_tr_sel, y_tr)
print(f"\nOriginal Acc : {accuracy_score(y_te, rf.predict(X_te)):.4f}")
print(f"Selected Acc : {accuracy_score(y_te, rf_sel.predict(X_te_sel)):.4f}")


## 3. XGBoost Importance + 상위 특성만 사용


In [ ]:
xgb_clf = xgb.XGBClassifier(n_estimators=100, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_tr, y_tr)
imp_xgb = pd.Series(xgb_clf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("XGBoost Top 10 features:")
print(imp_xgb.head(10))

top_n = 10
top_features = imp_xgb.head(top_n).index
idx = [list(feature_names).index(f) for f in top_features]
X_tr_top = X_tr[:, idx]
X_te_top = X_te[:, idx]
xgb_top = xgb.XGBClassifier(n_estimators=100, random_state=42, verbosity=0, use_label_encoder=False, eval_metric='logloss')
xgb_top.fit(X_tr_top, y_tr)
print(f"\nAll features Acc: {accuracy_score(y_te, xgb_clf.predict(X_te)):.4f}")
print(f"Top {top_n} Acc     : {accuracy_score(y_te, xgb_top.predict(X_te_top)):.4f}")
